# NBA Game-Prediction Dataset Builder (NBA API, last 10 seasons)

Replaces the SQLite-from-Kaggle source with **live data from `nba_api`**, so it stays current.

**Two phases:**

1. **Phase 1 — Box scores.** Fetch team-level game logs from the NBA stats API for each season, stack them, and produce `box_data.csv` in the same T1/T2 layout (T1=home, T2=away).
2. **Phase 2 — Pre-game features.** Identical leakage-free pipeline as the previous notebook: each game's features use only that team's prior games in the same season.

**Outputs:**
- `box_data.csv` — game-level T1/T2 box scores
- `matchups.csv` — pre-game season-to-date features for both teams + `Target_Win`

**Phase 1 caches each season's raw fetch to `raw_seasons/`** so you don't pay the network cost twice if you re-run.


## Setup

In [ ]:
# One-time install if you don't have it:
# !pip install nba_api pandas numpy

import time
from pathlib import Path

import numpy as np
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog

# ---- Config ----
# 10 seasons (last full decade). Edit if you want a different range.
SEASONS = [f"{y}-{str(y+1)[2:]}" for y in range(2016, 2026)]   # 2016-17 .. 2025-26
SEASON_TYPE = "Regular Season"   # or "Playoffs"
RAW_DIR     = Path("raw_seasons")
OUT_DIR     = Path(".")
RAW_DIR.mkdir(exist_ok=True)

SLEEP_BETWEEN_CALLS = 0.6        # be polite to stats.nba.com
RETRIES             = 4

# Same Elo constants as before
ELO_BASE      = 1500.0
ELO_K         = 20.0
ELO_HOME_ADV  = 100.0
ELO_REGRESS   = 0.25

STAT_COLS = ['Score','FGM','FGA','FGM3','FGA3','FTM','FTA',
             'OR','DR','Ast','TO','Stl','Blk','PF']
OPP_COLS  = [f'Opp_{s}' for s in STAT_COLS]

print("Seasons to fetch:", SEASONS)

## Phase 1 — Fetch & build `box_data.csv`

`LeagueGameLog` returns one row per (team, game), so 2 rows per game. We pivot home vs. away (the `MATCHUP` field uses `vs.` for home and `@` for away) and merge into one row per game with T1_*/T2_* columns.

### Helper: fetch one season with retries + cache

In [ ]:
def fetch_season(season: str,
                 season_type: str = SEASON_TYPE,
                 retries: int = RETRIES,
                 cache_dir: Path = RAW_DIR) -> pd.DataFrame:
    """Pull team game logs for a single season. Caches to a CSV in cache_dir."""
    fname = cache_dir / f"{season}_{season_type.replace(' ', '_')}.csv"
    if fname.exists():
        return pd.read_csv(fname)

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            log = leaguegamelog.LeagueGameLog(
                season=season,
                season_type_all_star=season_type,
                player_or_team_abbreviation='T',  # team-level
                timeout=60,
            )
            df = log.get_data_frames()[0]
            df.to_csv(fname, index=False)
            return df
        except Exception as e:
            last_err = e
            wait = 5 * attempt
            print(f"  [{season}] attempt {attempt}/{retries} failed: {e}; "
                  f"retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {season} after {retries} attempts: {last_err}")

### Run the fetch loop

In [ ]:
raw_frames = []
for s in SEASONS:
    print(f"Fetching {s} ...", end=" ", flush=True)
    df = fetch_season(s)
    print(f"{len(df):,} team-game rows")
    raw_frames.append(df)
    time.sleep(SLEEP_BETWEEN_CALLS)

raw = pd.concat(raw_frames, ignore_index=True)
print(f"\nTotal team-game rows: {len(raw):,}")
raw.head()

### Helper: stack team-game rows into the T1/T2 box layout

`MATCHUP` like `"LAL vs. BOS"` = home (T1), `"LAL @ BOS"` = away (T2). We split, rename columns, and inner-join on `GAME_ID`.

In [ ]:
# Map of nba_api column names -> our schema (without the T1_/T2_ prefix)
NBA_TO_OURS = {
    'TEAM_ID':   'TeamID',
    'TEAM_NAME': 'TeamName',
    'PTS':  'Score',
    'FGM':  'FGM',  'FGA':  'FGA',
    'FG3M': 'FGM3', 'FG3A': 'FGA3',
    'FTM':  'FTM',  'FTA':  'FTA',
    'OREB': 'OR',   'DREB': 'DR',
    'AST':  'Ast',  'TOV':  'TO',
    'STL':  'Stl',  'BLK':  'Blk',
    'PF':   'PF',
}


def build_box_data(raw: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Convert nba_api LeagueGameLog rows into the T1/T2 box layout."""
    df = raw.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    # SEASON_ID like '22015' -> Season = 2015 (start year of the 2015-16 season)
    df['Season'] = df['SEASON_ID'].astype(str).str[-4:].astype(int)

    # Keep only standard regular-season matchup strings ("X vs. Y" or "X @ Y").
    # All-Star, preseason oddities, etc. get dropped here.
    mask = df['MATCHUP'].str.contains(r' vs\. | @ ', regex=True, na=False)
    n_dropped = (~mask).sum()
    if n_dropped and verbose:
        print(f"  dropped {n_dropped} rows with non-standard MATCHUP")
    df = df[mask].copy()
    df['IS_HOME'] = df['MATCHUP'].str.contains(' vs. ', regex=False)

    # First-pass dedupe: same (game, team) appearing twice in the API response
    before = len(df)
    df = df.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])
    if verbose and len(df) < before:
        print(f"  dropped {before - len(df)} duplicate (GAME_ID, TEAM_ID) rows")

    # Drop rows missing core box-score numbers
    needed = list(NBA_TO_OURS.keys()) + ['WL']
    df = df.dropna(subset=needed)

    # Diagnose any GAME_IDs that don't have exactly 1 home + 1 away row.
    # This catches rare data glitches that would otherwise blow up the merge.
    counts = df.groupby('GAME_ID')['IS_HOME'].agg(home_rows='sum', total='count')
    bad = counts[(counts['home_rows'] != 1) | (counts['total'] != 2)]
    if len(bad) > 0:
        if verbose:
            print(f"  dropping {len(bad)} games with unusual home/away structure")
            print(f"    example GAME_IDs: {list(bad.index[:5])}")
        df = df[~df['GAME_ID'].isin(bad.index)]

    cols_to_keep = ['GAME_ID', 'GAME_DATE', 'Season', 'WL'] + list(NBA_TO_OURS.keys())

    home = (df[df['IS_HOME']][cols_to_keep]
            .rename(columns={**{k: f'T1_{v}' for k, v in NBA_TO_OURS.items()},
                             'WL': 'T1_WL'}))
    away = (df[~df['IS_HOME']][['GAME_ID', 'WL'] + list(NBA_TO_OURS.keys())]
            .rename(columns={**{k: f'T2_{v}' for k, v in NBA_TO_OURS.items()},
                             'WL': 'T2_WL'}))

    # Clean enough now that an inner merge gives exactly the games we want.
    box = home.merge(away, on='GAME_ID', how='inner')

    box = box.rename(columns={'GAME_ID': 'GameID', 'GAME_DATE': 'DayDate'})
    box['T1_Loc']     = 'H'
    box['Target_Win'] = (box['T1_WL'] == 'W').astype(int)
    box = box.drop(columns=['T1_WL', 'T2_WL'])

    # Order columns the same way the SQLite version did
    front = ['Season', 'DayDate', 'GameID',
             'T1_TeamID', 'T1_TeamName', 'T2_TeamID', 'T2_TeamName', 'T1_Loc']
    t1_stats = [f'T1_{c}' for c in STAT_COLS]
    t2_stats = [f'T2_{c}' for c in STAT_COLS]
    box = box[front + t1_stats + t2_stats + ['Target_Win']]

    box = box.sort_values(['DayDate', 'GameID']).reset_index(drop=True)
    return box

In [ ]:
box = build_box_data(raw)
box.to_csv(OUT_DIR / "box_data.csv", index=False)
print(f"Wrote box_data.csv ({len(box):,} games, "
      f"seasons {box['Season'].min()}-{box['Season'].max()})")
box.head()

## Phase 2 — Pre-game features

Identical leakage-free pipeline from before. For every `(GameID, TeamID)`, compute season-to-date averages, win pcts, advanced metrics, last-14-day form, pre-game Elo, and a snapshot seed — all using **only games strictly before** the current game in the same season.

The first game of each team-season has no priors, so its features come out as `NaN`.

### Helpers

In [ ]:
def explode_to_team_perspective(box: pd.DataFrame) -> pd.DataFrame:
    """Each game becomes 2 rows -- one from each team's perspective."""
    def side(prefix_self, prefix_opp, loc):
        d = pd.DataFrame({
            'Season' : box['Season'],
            'DayDate': box['DayDate'],
            'GameID' : box['GameID'],
            'TeamID' : box[f'{prefix_self}_TeamID'],
            'OppID'  : box[f'{prefix_opp}_TeamID'],
            'Loc'    : loc,
            'Won'    : box['Target_Win'] if prefix_self == 'T1' else 1 - box['Target_Win'],
        })
        for s in STAT_COLS:
            d[s]          = box[f'{prefix_self}_{s}']
            d[f'Opp_{s}'] = box[f'{prefix_opp}_{s}']
        return d

    return pd.concat([side('T1', 'T2', 'H'),
                      side('T2', 'T1', 'A')], ignore_index=True)


def compute_possessions(g: pd.DataFrame) -> pd.DataFrame:
    """Per-game possessions (Poss = FGA - OR + TO + 0.475*FTA)."""
    g = g.copy()
    g['Poss']    = g['FGA']     - g['OR']     + g['TO']     + 0.475 * g['FTA']
    g['OppPoss'] = g['Opp_FGA'] - g['Opp_OR'] + g['Opp_TO'] + 0.475 * g['Opp_FTA']
    return g

In [ ]:
def compute_elo_pregame(box: pd.DataFrame,
                        base: float = ELO_BASE,
                        k: float = ELO_K,
                        hca: float = ELO_HOME_ADV,
                        regress: float = ELO_REGRESS) -> pd.DataFrame:
    """One row per game with both teams' pre-game Elo."""
    elo: dict = {}
    records = []
    prev_season = None

    for _, row in box[['Season','GameID','T1_TeamID','T2_TeamID','Target_Win']].iterrows():
        season = row['Season']
        if prev_season is not None and season != prev_season:
            for tid in elo:
                elo[tid] = (1 - regress) * elo[tid] + regress * base
        prev_season = season

        t1, t2 = row['T1_TeamID'], row['T2_TeamID']
        e1, e2 = elo.get(t1, base), elo.get(t2, base)

        # snapshot BEFORE the game updates Elo
        records.append({
            'GameID'        : row['GameID'],
            'T1_TeamID'     : t1,
            'T2_TeamID'     : t2,
            'T1_Pregame_Elo': e1,
            'T2_Pregame_Elo': e2,
        })

        exp1   = 1.0 / (1.0 + 10 ** (-((e1 + hca) - e2) / 400.0))
        delta  = k * (row['Target_Win'] - exp1)
        elo[t1] = e1 + delta
        elo[t2] = e2 - delta

    return pd.DataFrame(records)

### Main computation

In [ ]:
def compute_pregame_features(box: pd.DataFrame) -> pd.DataFrame:
    tg = explode_to_team_perspective(box)
    tg = compute_possessions(tg)
    tg = tg.sort_values(['TeamID', 'Season', 'DayDate', 'GameID']).reset_index(drop=True)

    grp_keys = ['TeamID', 'Season']
    grp = tg.groupby(grp_keys)

    # ---- Games played BEFORE this one (0 for first game of season) ----
    tg['Games_Played'] = grp.cumcount()

    # ---- Cumulative totals BEFORE this game (cumsum - current) ----
    sum_cols = STAT_COLS + OPP_COLS + ['Poss', 'OppPoss']
    for c in sum_cols:
        tg[f'cum_{c}'] = grp[c].cumsum() - tg[c]
    tg['cum_Wins'] = grp['Won'].cumsum() - tg['Won']

    # ---- Per-game averages ----
    safe_games = tg['Games_Played'].replace(0, np.nan)
    for c in STAT_COLS + OPP_COLS:
        tg[f'Avg_{c}'] = tg[f'cum_{c}'] / safe_games

    # ---- Advanced metrics on cumulative totals ----
    safe_poss     = tg['cum_Poss'].replace(0, np.nan)
    safe_opp_poss = tg['cum_OppPoss'].replace(0, np.nan)
    safe_fga      = tg['cum_FGA'].replace(0, np.nan)
    safe_or_dr    = (tg['cum_OR'] + tg['cum_Opp_DR']).replace(0, np.nan)

    tg['Avg_Off_Eff'] = (tg['cum_Score']     / safe_poss)     * 100   # OffEff
    tg['Avg_Def_Eff'] = (tg['cum_Opp_Score'] / safe_opp_poss) * 100   # DefEff
    tg['Net_Rating']  = tg['Avg_Off_Eff'] - tg['Avg_Def_Eff']         # NetEff
    tg['EFG_Pct']     = (tg['cum_FGM'] + 0.5 * tg['cum_FGM3']) / safe_fga
    tg['TO_Rate']     = tg['cum_TO'] / safe_poss
    tg['OR_Pct']      = tg['cum_OR'] / safe_or_dr

    # ---- Win pct (overall / home / away) BEFORE current game ----
    tg['Win_Pct']         = tg['cum_Wins'] / safe_games
    tg['Overall_Win_Pct'] = tg['Win_Pct']

    tg['_is_home']  = (tg['Loc'] == 'H').astype(int)
    tg['_is_away']  = (tg['Loc'] == 'A').astype(int)
    tg['_home_won'] = tg['Won'] * tg['_is_home']
    tg['_away_won'] = tg['Won'] * tg['_is_away']
    g2 = tg.groupby(grp_keys)
    cum_hg = g2['_is_home'].cumsum()  - tg['_is_home']
    cum_hw = g2['_home_won'].cumsum() - tg['_home_won']
    cum_ag = g2['_is_away'].cumsum()  - tg['_is_away']
    cum_aw = g2['_away_won'].cumsum() - tg['_away_won']
    tg['Home_Win_Pct'] = cum_hw / cum_hg.replace(0, np.nan)
    tg['Away_Win_Pct'] = cum_aw / cum_ag.replace(0, np.nan)
    tg = tg.drop(columns=['_is_home', '_is_away', '_home_won', '_away_won'])

    # ---- Last-14-day win pct (rolling time window, current row excluded) ----
    last14 = np.full(len(tg), np.nan)
    for _, idx in tg.groupby(grp_keys, sort=False).indices.items():
        sub = tg.iloc[idx]
        s = pd.Series(sub['Won'].values,
                      index=pd.DatetimeIndex(sub['DayDate']))
        last14[idx] = s.rolling('14D', closed='left').mean().values
    tg['Last_14_Days_Win_Pct'] = last14

    # ---- Pre-game Elo (merged from chronological walk) ----
    elo_df = compute_elo_pregame(box)
    elo_t1 = elo_df[['GameID','T1_TeamID','T1_Pregame_Elo']].rename(
        columns={'T1_TeamID': 'TeamID', 'T1_Pregame_Elo': 'Pregame_Elo'})
    elo_t2 = elo_df[['GameID','T2_TeamID','T2_Pregame_Elo']].rename(
        columns={'T2_TeamID': 'TeamID', 'T2_Pregame_Elo': 'Pregame_Elo'})
    elo_long = pd.concat([elo_t1, elo_t2], ignore_index=True)
    tg = tg.merge(elo_long, on=['GameID', 'TeamID'], how='left')

    # ---- Seed: rank by current Win_Pct within season as of game date ----
    seed_parts = []
    for season, sub in tg.groupby('Season'):
        pvt = (sub.pivot_table(index='DayDate', columns='TeamID',
                               values='Win_Pct', aggfunc='mean')
                  .sort_index().ffill())
        ranks = pvt.rank(axis=1, method='min', ascending=False)
        long = (ranks.reset_index()
                     .melt(id_vars='DayDate', var_name='TeamID', value_name='seed'))
        long['Season'] = season
        seed_parts.append(long)
    seeds = pd.concat(seed_parts, ignore_index=True)
    tg = tg.merge(seeds, on=['Season','TeamID','DayDate'], how='left')

    keep = (['Season','DayDate','GameID','TeamID','OppID','Loc','Won','Games_Played']
            + [f'Avg_{c}' for c in STAT_COLS + OPP_COLS]
            + ['Avg_Off_Eff','Avg_Def_Eff','Net_Rating','EFG_Pct','TO_Rate','OR_Pct',
               'Win_Pct','Overall_Win_Pct','Home_Win_Pct','Away_Win_Pct',
               'Last_14_Days_Win_Pct','Pregame_Elo','seed'])
    return tg[keep].reset_index(drop=True)

In [ ]:
pregame = compute_pregame_features(box)
print(f"Pre-game features: {len(pregame):,} rows ({len(pregame)//2:,} games x 2 teams)")
print(f"Rows with NaN features (first game of each team-season): "
      f"{pregame['Avg_Score'].isna().sum():,}")
pregame.head()

## Phase 3 — Build `matchups.csv`

Merge each game with both teams' pre-game features, in the column order from your spec (with `End_Season_Elo` -> `Pregame_Elo`, since the Elo is now snapshotted before each game rather than at season-end).

In [ ]:
FEATURE_COLS = [
    'Avg_Off_Eff', 'Avg_Def_Eff', 'Net_Rating',
    'Win_Pct', 'Pregame_Elo',
    'Overall_Win_Pct', 'Home_Win_Pct', 'Away_Win_Pct',
    'Avg_Score', 'Avg_FGM', 'Avg_FGA', 'Avg_FGM3', 'Avg_FGA3',
    'Avg_FTM',   'Avg_FTA', 'Avg_OR',  'Avg_DR',
    'Avg_Ast',   'Avg_TO',  'Avg_Stl', 'Avg_Blk', 'Avg_PF',
    'Avg_Opp_Score', 'Avg_Opp_FGM', 'Avg_Opp_FGA',
    'Avg_Opp_FGM3',  'Avg_Opp_FGA3',
    'Avg_Opp_FTM',   'Avg_Opp_FTA',
    'Avg_Opp_OR',    'Avg_Opp_DR',   'Avg_Opp_Ast',
    'Avg_Opp_TO',    'Avg_Opp_Stl',  'Avg_Opp_Blk', 'Avg_Opp_PF',
    'seed', 'Last_14_Days_Win_Pct',
]


def build_matchups(box: pd.DataFrame, pregame: pd.DataFrame) -> pd.DataFrame:
    feats = pregame[['GameID', 'TeamID'] + FEATURE_COLS]

    t1 = feats.rename(columns={'TeamID': 'T1_TeamID',
                               **{c: f'T1_{c}' for c in FEATURE_COLS}})
    t2 = feats.rename(columns={'TeamID': 'T2_TeamID',
                               **{c: f'T2_{c}' for c in FEATURE_COLS}})

    m = (box[['Season','DayDate','GameID','T1_TeamID','T2_TeamID','Target_Win']]
            .merge(t1, on=['GameID', 'T1_TeamID'])
            .merge(t2, on=['GameID', 'T2_TeamID']))

    m['ID'] = (m['Season'].astype(str) + '_'
               + m['T1_TeamID'].astype(str) + '_'
               + m['T2_TeamID'].astype(str))

    ordered = (['T1_TeamID', 'T2_TeamID', 'Season', 'DayDate', 'GameID', 'ID']
               + [f'T1_{c}' for c in FEATURE_COLS]
               + [f'T2_{c}' for c in FEATURE_COLS]
               + ['Target_Win'])
    return m[ordered]

In [ ]:
matchups = build_matchups(box, pregame)
matchups.to_csv(OUT_DIR / "matchups.csv", index=False)
print(f"Wrote matchups.csv ({len(matchups):,} rows, {len(matchups.columns)} cols)")

# Optional: drop the first-game-of-season rows that have NaN features
clean = matchups.dropna(subset=[c for c in matchups.columns
                                if c.startswith(('T1_Avg', 'T2_Avg'))])
print(f"Rows after dropping NaN-feature games: {len(clean):,}")
matchups.head()

## Sanity checks

Same checks as before — `Games_Played` should start at 0 and increment by 1, and `Avg_Score` for any given pre-game row should equal the manual mean of that team's prior games' scores.

In [ ]:
gp_check = (pregame.sort_values(['TeamID','Season','DayDate','GameID'])
                   .groupby(['TeamID','Season'])['Games_Played']
                   .agg(['min','max','count']))
print("Games_Played min should be 0 for every team-season:",
      (gp_check['min'] == 0).all())
print("Games_Played max should equal count - 1:",
      (gp_check['max'] == gp_check['count'] - 1).all())

team_log = (explode_to_team_perspective(box)
            .sort_values(['TeamID','Season','DayDate','GameID']))
sample = (pregame[pregame['Games_Played'] >= 4]
          .sort_values(['TeamID','Season','DayDate','GameID'])
          .head(1).iloc[0])

prior = team_log[(team_log['TeamID'] == sample['TeamID']) &
                 (team_log['Season'] == sample['Season']) &
                 (team_log['DayDate'] < sample['DayDate'])]
print(f"\nSpot check for TeamID={sample['TeamID']}, Season={sample['Season']}, "
      f"GameID={sample['GameID']}")
print(f"  Pre-game Avg_Score in pregame:        {sample['Avg_Score']:.3f}")
print(f"  Manual mean of prior games' Score:    {prior['Score'].mean():.3f}")

## Done

You now have:

- `box_data.csv` — game-level T1/T2 box scores from the last ~10 NBA seasons
- `matchups.csv` — pre-game features for both teams, ready for modeling

`raw_seasons/` holds per-season caches in case you want to refresh just the current season later.

**To refresh the current season**: delete `raw_seasons/2025-26_Regular_Season.csv` and re-run the fetch loop. The other seasons stay cached.
